In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager, rc
import seaborn as sns
import numpy as np

from konlpy.tag import Okt
from sklearn.preprocessing import OneHotEncoder, LabelBinarizer
from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# matplotlib의 한글문제를 해결
font_name = font_manager.FontProperties(fname="c:/Windows/Fonts/malgun.ttf").get_name()
# font_name
rc('font', family=font_name)

In [ ]:
# preprocess_text 함수 적용된 데이터 (시간이 오래걸려 한번 수행 후 csv파일로 저장해둠.)
df = pd.read_csv('dataset/Pre-processing_news.csv')
df

,PressCompany,Category,Document
0,뉴스1,정치,독도 일본 땅 주장 역사 국제 법 고유 영토 상보 정부 강력히 항의 한일 관계 도움...
1,파이낸셜뉴스,정치,김정은 요새 순항미사일 쏴 대는 까닭 주일 사이 차례 순항미사일 도발 반길주 유엔 ...
2,오마이뉴스,정치,듣도 보도 못 조선 핼로윈 특별법 이태원 참사 특별법 핼로윈 특별법 이라 칭해 정식...
3,연합뉴스,정치,이재명 오늘 신년 회견 총선 각오 밝히고 민주당 지지 호소 정권 비판 하며 대안 제...
4,국제신문,정치,속보 대통령 이태원 특별법 거부권 행사 취임 후 윤석열 대통령 이태원 특별법 이태원...
...,...,...,...
6967,블로터,IT/과학,흑자 달성 디스플레이 과제 재무 건전 디스플레이 파주 공장 전경 사진 디스플레이 예...
6968,머니투데이,IT/과학,하반신 마비 쥐 신약 맞고 걸었다 영상 다리 근육 뻣뻣해지다가 마비 되는 질환 생명...
6969,블로터,IT/과학,경영 불안 요기 이정환 대표 사임 설 마케팅 확대 제동 불가피 이정환 요기 대표 사...
6970,디지털타임스,IT/과학,삼바 영업 익 첫 돌파 바이오 연대기 획 국내 제약 바이오 기업 중 최초 화이자 위...


### 변수 생성 (TF-IDF(본문), Word2Vec(본문), Label(카테고리))

In [9]:
# TF-IDF ; 본문
tfidf_vectorizer = TfidfVectorizer(max_features=20000)

# 학습 데이터에 대해 TF-IDFVectorizer 학습 및 변환
document_tfidf = tfidf_vectorizer.fit_transform(df['Document']).todense()

In [4]:
import gensim
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize

# 데이터프레임에서 'Document' 열을 가져와서 텍스트 데이터를 리스트로 변환
documents = df['Document'].tolist()

# 각 문서를 토큰화하여 리스트로 변환
tokenized_documents = [word_tokenize(document) for document in documents]

# Word2Vec 모델 생성
model = Word2Vec(sentences=tokenized_documents, vector_size=100, window=5, min_count=1, sg=0)

# 모델 학습
model.train(tokenized_documents, total_examples=len(tokenized_documents), epochs=10)

# 각 문서의 단어 벡터를 평균하여 고정 크기의 벡터로 변환
document_vectors_np = np.array([np.mean(model.wv[words], axis=0) for words in tokenized_documents])

In [5]:
# Label : 카테고리 (target data)
label_encoder = LabelEncoder()
df['CategoryEncoded'] = label_encoder.fit_transform(df['Category'])

### 함수 생성

In [6]:
def split_data(document):
    # 학습용 데이터와 테스트용 데이터로 나누기
    x_train, x_test, y_train, y_test = train_test_split(
        np.array(document),
        np.array(df['CategoryEncoded']),  # 타겟 데이터
        test_size=0.3,  # 테스트 데이터의 비율 설정
        random_state=23  # 랜덤 시드 설정 (재현성을 위해)
    )

    # 분리된 데이터 확인
    print("학습 세트 크기:", x_train.shape, y_train.shape)
    print("테스트 세트 크기:", x_test.shape, y_test.shape)
    
    return x_train, x_test, y_train, y_test

In [7]:
def fit_model(model, x_train, x_test, y_train, y_test):
    # 모델 학습
    model.fit(x_train, y_train)
    
    # 테스트 데이터에 대한 예측
    y_pred = model.predict(x_test)

    # 정확도 스코어 계산
    accuracy = accuracy_score(y_test, y_pred)
    print("정확도:", accuracy)

    # 분류 보고서 출력
    report = classification_report(y_test, y_pred)
    print("분류 보고서:\n", report)

### 다중 로지스틱 회귀 (TF-IDF(본문), Label(카테고리)) 적용

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

x_train, x_test, y_train, y_test = split_data(document_tfidf)

# 로지스틱 회귀 모델 초기화
logistic_regression = LogisticRegression(multi_class='multinomial', max_iter=1000)

fit_model(logistic_regression, x_train, x_test, y_train, y_test)

학습 세트 크기: (4880, 20000) (4880,)
테스트 세트 크기: (2092, 20000) (2092,)
정확도: 0.859942638623327
분류 보고서:
               precision    recall  f1-score   support

           0       0.85      0.81      0.83       442
           1       0.84      0.83      0.84       414
           2       0.85      0.84      0.84       404
           3       0.81      0.88      0.84       426
           4       0.96      0.95      0.95       406

    accuracy                           0.86      2092
   macro avg       0.86      0.86      0.86      2092
weighted avg       0.86      0.86      0.86      2092



### 다중 로지스틱 회귀 (Word2Vec(본문), Label(카테고리)) 적용

In [23]:
x_train, x_test, y_train, y_test = split_data(document_vectors_np)

# 로지스틱 회귀 모델 초기화
logistic_regression = LogisticRegression(multi_class='multinomial', max_iter=1000)

fit_model(logistic_regression, x_train, x_test, y_train, y_test)

학습 세트 크기: (4880, 100) (4880,)
테스트 세트 크기: (2092, 100) (2092,)
정확도: 0.7901529636711281
분류 보고서:
               precision    recall  f1-score   support

           0       0.77      0.74      0.76       442
           1       0.72      0.75      0.74       414
           2       0.77      0.78      0.78       404
           3       0.76      0.76      0.76       426
           4       0.94      0.92      0.93       406

    accuracy                           0.79      2092
   macro avg       0.79      0.79      0.79      2092
weighted avg       0.79      0.79      0.79      2092



### 나이브베이지안 모델 (TF-IDF(본문), Label(카테고리)) 적용

In [24]:
from sklearn.naive_bayes import MultinomialNB

x_train, x_test, y_train, y_test = split_data(document_tfidf)

# 나이브 베이지안 모델 생성
naive_bayes_classifier = MultinomialNB()

fit_model(naive_bayes_classifier, x_train, x_test, y_train, y_test)

학습 세트 크기: (4880, 20000) (4880,)
테스트 세트 크기: (2092, 20000) (2092,)
정확도: 0.8307839388145315
분류 보고서:
               precision    recall  f1-score   support

           0       0.83      0.81      0.82       442
           1       0.75      0.87      0.81       414
           2       0.85      0.77      0.81       404
           3       0.85      0.73      0.79       426
           4       0.89      0.97      0.93       406

    accuracy                           0.83      2092
   macro avg       0.83      0.83      0.83      2092
weighted avg       0.83      0.83      0.83      2092



### 가우시안NB 모델 (Word2Vec(본문), Label(카테고리)) 적용

In [26]:
from sklearn.naive_bayes import GaussianNB

x_train, x_test, y_train, y_test = split_data(document_vectors_np)

# 가우시안NB 모델 생성
gaussian_naive_bayes_classifier = GaussianNB()

fit_model(gaussian_naive_bayes_classifier, x_train, x_test, y_train, y_test)

학습 세트 크기: (4880, 100) (4880,)
테스트 세트 크기: (2092, 100) (2092,)
정확도: 0.7151051625239006
분류 보고서:
               precision    recall  f1-score   support

           0       0.64      0.64      0.64       442
           1       0.65      0.75      0.70       414
           2       0.71      0.69      0.70       404
           3       0.73      0.69      0.71       426
           4       0.87      0.82      0.84       406

    accuracy                           0.72      2092
   macro avg       0.72      0.72      0.72      2092
weighted avg       0.72      0.72      0.72      2092



### XGB (TF-IDF(본문), Label(카테고리)) 적용

In [27]:
from xgboost import XGBClassifier
x_train, x_test, y_train, y_test = split_data(document_tfidf)

# XGBoost 분류기 생성
xgb_classifier = XGBClassifier(
    n_estimators=100,  # 부스팅 라운드 수
    max_depth=6,       # 트리 최대 깊이
    learning_rate=0.1, # 학습률
    random_state=42,    # 난수 시드
    use_label_encoder=False,
    eval_metric='merror'  # 여기에 'merror'를 설정
)

fit_model(xgb_classifier, x_train, x_test, y_train, y_test)

학습 세트 크기: (4880, 20000) (4880,)
테스트 세트 크기: (2092, 20000) (2092,)
정확도: 0.8460803059273423
분류 보고서:
               precision    recall  f1-score   support

           0       0.82      0.82      0.82       442
           1       0.82      0.81      0.82       414
           2       0.83      0.82      0.82       404
           3       0.83      0.83      0.83       426
           4       0.93      0.96      0.95       406

    accuracy                           0.85      2092
   macro avg       0.85      0.85      0.85      2092
weighted avg       0.85      0.85      0.85      2092



In [ ]:
from sklearn.model_selection import GridSearchCV

# XGBoost 분류기 생성
xgb_classifier = XGBClassifier(use_label_encoder=False, eval_metric='merror')

# 파라미터 그리드 설정
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 6, 9],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.5, 0.7, 1.0]
}

# GridSearchCV 생성
grid_search = GridSearchCV(xgb_classifier, param_grid, cv=3, verbose=2)

grid_search.fit(x_train, y_train)

# 최적의 파라미터와 점수 출력
print("Best parameters:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)

# 테스트 데이터에 대한 성능 평가
best_model = grid_search.best_estimator_
predictions = best_model.predict(x_test)

# 성능 평가 결과 출력
print(classification_report(y_test, predictions))


Fitting 3 folds for each of 81 candidates, totalling 243 fits
[CV] END learning_rate=0.01, max_depth=3, n_estimators=50, subsample=0.5; total time= 1.7min
[CV] END learning_rate=0.01, max_depth=3, n_estimators=50, subsample=0.5; total time= 2.1min
[CV] END learning_rate=0.01, max_depth=3, n_estimators=50, subsample=0.5; total time= 2.2min
[CV] END learning_rate=0.01, max_depth=3, n_estimators=50, subsample=0.7; total time= 2.1min
[CV] END learning_rate=0.01, max_depth=3, n_estimators=50, subsample=0.7; total time= 2.0min
[CV] END learning_rate=0.01, max_depth=3, n_estimators=50, subsample=0.7; total time= 2.0min
[CV] END learning_rate=0.01, max_depth=3, n_estimators=50, subsample=1.0; total time= 1.9min
[CV] END learning_rate=0.01, max_depth=3, n_estimators=50, subsample=1.0; total time= 1.9min
[CV] END learning_rate=0.01, max_depth=3, n_estimators=50, subsample=1.0; total time= 1.9min
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=0.5; total time= 3.6min
[CV] EN

### SVM (TF-IDF(본문), Label(카테고리)) 적용

In [10]:
from sklearn.svm import SVC

x_train, x_test, y_train, y_test = split_data(document_tfidf)

# SVM 분류기 생성
svm_classifier = SVC()

fit_model(svm_classifier, x_train, x_test, y_train, y_test)

학습 세트 크기: (4880, 20000) (4880,)
테스트 세트 크기: (2092, 20000) (2092,)
정확도: 0.8589866156787763
분류 보고서:
               precision    recall  f1-score   support

           0       0.85      0.82      0.83       442
           1       0.84      0.83      0.84       414
           2       0.84      0.83      0.84       404
           3       0.80      0.88      0.84       426
           4       0.97      0.94      0.96       406

    accuracy                           0.86      2092
   macro avg       0.86      0.86      0.86      2092
weighted avg       0.86      0.86      0.86      2092



### SVM (Word2Vec(본문), Label(카테고리)) 적용

In [11]:
from sklearn.svm import SVC

x_train, x_test, y_train, y_test = split_data(document_vectors_np)

# SVM 분류기 생성
svm_classifier = SVC()

fit_model(svm_classifier, x_train, x_test, y_train, y_test)

학습 세트 크기: (4880, 100) (4880,)
테스트 세트 크기: (2092, 100) (2092,)
정확도: 0.8350860420650096
분류 보고서:
               precision    recall  f1-score   support

           0       0.79      0.79      0.79       442
           1       0.80      0.79      0.80       414
           2       0.81      0.83      0.82       404
           3       0.83      0.81      0.82       426
           4       0.95      0.96      0.95       406

    accuracy                           0.84      2092
   macro avg       0.84      0.84      0.84      2092
weighted avg       0.84      0.84      0.84      2092



### SVM (Word2Vec(본문), Label(카테고리)) 적용 -> GridSearchCV로 최적 파라미터 찾기

In [12]:
from sklearn import model_selection, datasets, metrics

param_grid = {'C' : [0.1, 1, 10, 100, 1000], 
             'gamma' : [1, 0.1, 0.01, 0.001, 0.0001],
             'kernel' : ['rbf']}

grid = model_selection.GridSearchCV(SVC(), param_grid, refit=True, verbose=1, cv=5)
grid.fit(x_train, y_train)
print('\nThe best parameters are ', grid.best_params_)

grid_predictions = grid.predict(x_test)

print()
print(metrics.classification_report(y_test, grid_predictions)) 
print("훈련 세트 정확도: {:.3f}".format(grid.score(x_train, y_train)))
print("테스트 세트 정확도: {:.3f}".format(grid.score(x_test, y_test)))

Fitting 5 folds for each of 25 candidates, totalling 125 fits

The best parameters are  {'C': 10, 'gamma': 0.1, 'kernel': 'rbf'}

              precision    recall  f1-score   support

           0       0.80      0.84      0.82       442
           1       0.83      0.81      0.82       414
           2       0.85      0.83      0.84       404
           3       0.84      0.82      0.83       426
           4       0.95      0.96      0.95       406

    accuracy                           0.85      2092
   macro avg       0.85      0.85      0.85      2092
weighted avg       0.85      0.85      0.85      2092

훈련 세트 정확도: 0.988
테스트 세트 정확도: 0.852
